# 04. Raw 64-Channel Spectral Unmixing Complement (`korotkaja_2024_breast_tumor_tme`)

Because the public spectral PBMC datasets in `data/spectral/` were deposited after SpectroFlo unmixing, this tutorial complements **[03. Spectral 32-Color PBMC Cohort](03_spectral_pbmc_cohort_walkthrough.ipynb)** using `korotkaja_2024_breast_tumor_tme` (murine 4T1 breast tumor TME, 6 samples + 8 single-stain controls + unstained cells) — our flagship **raw 64-detector Cytek Aurora APD** dataset.

**What you will learn:**
1. How `flowx unmix` / `flowx pipeline` builds a $D \times P$ reference spectral library from `raw/controls/`.
2. How single-component autofluorescence (`AF`) extraction separates tissue autofluorescence from fluorophores.
3. How to audit Cosine Spectral Similarity, Condition Number, Unmixing Residuals, and $N \times N$ unmixed plots.

> **Prerequisites (Download Raw `.fcs` Data):** Curated `metadata/` CSVs are included in the repository. Before running this notebook, download the raw `.fcs` files once from the terminal:
> ```bash
> flowx download korotkaja_2024_breast_tumor_tme
> ```

In [1]:
import os
from pathlib import Path

# Ensure working directory is the repository root whether launched from root or docs/tutorials/
if not Path("configs/config_run").exists() and Path("../../configs/config_run").exists():
    os.chdir("../..")
print("Working directory ready:", Path("configs/config_run").exists())

Working directory ready: True


## 1. Inspect the 64-Detector Config & Run Spectral Unmixing

You can run spectral unmixing standalone via `flowx unmix --config ...` or end-to-end via `flowx pipeline`:

```bash
# Standalone spectral unmixing
flowx unmix --config configs/config_run/spectral/korotkaja_2024_breast_tumor_tme.yaml

# Or full end-to-end pipeline (~30 seconds)
flowx pipeline -c configs/config_run/spectral/korotkaja_2024_breast_tumor_tme.yaml --overwrite
```

In [2]:
import yaml
import pandas as pd
from pathlib import Path

cfg = yaml.safe_load(Path("configs/config_run/spectral/korotkaja_2024_breast_tumor_tme.yaml").read_text())
print("02_compensation config:", cfg["02_compensation"])

out_dir = Path("outputs/config_run/spectral/korotkaja_2024_breast_tumor_tme")
sig_df = pd.read_csv(out_dir / "02_compensation/tables/spectral_signature_matrix.csv", index_col=0)
print(f"\nReference Spectral Library shape: {sig_df.shape[0]} detectors x {sig_df.shape[1]} endmembers (incl. AF)")
print("Endmembers:", list(sig_df.columns))

02_compensation config: {'enabled': True, 'source': 'controls', 'controls_dir': 'data/spectral/korotkaja_2024_breast_tumor_tme/raw/controls', 'unmixing_method': 'ols', 'autofluorescence_mode': 'single'}

Reference Spectral Library shape: 64 detectors x 9 endmembers (incl. AF)
Endmembers: ['BV510', 'APC', 'PE-eFluor_610', 'PE', 'Real_Blue_RB_545', 'BV605', 'Live-or-dye_TM', 'FITC', 'Autofluorescence']


## 2. Inspect Spectral Similarity Matrix & Unmixing Diagnostics

Pairwise cosine similarity quantifies spectral overlap across all fluorophores and extracted cellular autofluorescence:

In [3]:
sim_df = pd.read_csv(out_dir / "02_compensation/tables/spectral_similarity_matrix.csv", index_col=0)
print(sim_df.round(3).to_string())

                  BV510    APC  PE-eFluor_610     PE  Real_Blue_RB_545  BV605  Live-or-dye_TM   FITC  Autofluorescence
BV510             1.000  0.012          0.039  0.073             0.426  0.323           0.296  0.283             0.660
APC               0.012  1.000          0.162  0.041             0.018  0.179           0.009  0.002             0.493
PE-eFluor_610     0.039  0.162          1.000  0.516             0.122  0.658           0.010  0.015             0.313
PE                0.073  0.041          0.516  1.000             0.178  0.326           0.011  0.027             0.200
Real_Blue_RB_545  0.426  0.018          0.122  0.178             1.000  0.231           0.066  0.629             0.309
BV605             0.323  0.179          0.658  0.326             0.231  1.000           0.043  0.049             0.512
Live-or-dye_TM    0.296  0.009          0.010  0.011             0.066  0.043           1.000  0.078             0.443
FITC              0.283  0.002          0.015  0

![Normalized Spectral Ribbons (Korotkaja 2024)](../images/tutorials/spectral_ribbons_korotkaja.png)

![Cosine Spectral Similarity Matrix (Korotkaja 2024)](../images/tutorials/spectral_similarity_korotkaja.png)

![Deconvolution Goodness-of-Fit Residuals (Korotkaja 2024)](../images/tutorials/spectral_gof_korotkaja.png)

![Post-Unmixing NxN Matrix (Korotkaja 2024)](../images/tutorials/spectral_nxn_korotkaja.png)

## 3. Downstream Discovery Summary

Once unmixed into biological fluorophore abundances, the cohort flows seamlessly into `Live-or-dye_TM` viability cleanup, `arcsinh(cofactor=2000)`, FlowSOM ($k=16 \to 3$ populations), UMAP, and MIFlowCyt reporting:

In [4]:
pop_map = pd.read_csv(out_dir / "05_population_discovery/tables/cluster_to_population_map.csv")
print(pop_map.groupby("Annotated_Population")[["Event_Count", "Pct_Total_Events"]].sum().to_string())

                                         Event_Count  Pct_Total_Events
Annotated_Population                                                  
CD3+ DN / Other T Cells                        72469            37.452
HLA-DR+ APC / Monocyte / B-Cell Lineage        14300             7.391
Murine Macrophages (F4/80+)                   106725            55.156


## References & Scientific Standards

* **Spectral Flow Cytometry Overview:** [](doi:10.1002/cyto.a.23779)
* **OMIP-098 39-Color Full Spectrum Panel:** [](doi:10.1002/cyto.a.24789)
* **Spillover Spreading Matrix (SSM):** [](doi:10.1002/cyto.a.22251)
* **MIFlowCyt Standard:** [](doi:10.1002/cyto.a.20623)
* **FlowSOM Clustering:** [](doi:10.1002/cyto.a.22625)